# xy — demo

Millions of points, interactive, in a notebook. Pan by dragging, zoom with the wheel, double-click to reset.
Shift-drag a scatter to box-select. One declarative API over one engine: the Reflex-flavored `xy.scatter_chart(...)` composition.

In [ ]:
import numpy as np

import xy
import xy.kernels

rng = np.random.default_rng(0)
print("kernel backend:", xy.kernels.BACKEND)

kernel backend: native


In [ ]:
models = [
    "GPT-5.6 Sol Ultra",
    "GPT-5.6 Sol",
    "Claude Mythos 5",
    "GPT-5.6 Terra",
    "Claude Fable 5",
    "GPT-5.5",
    "GPT-5.6 Luna",
    "Claude Opus 4.8",
    "Gemini 3.1 Pro Preview",
]
scores = [91.9, 88.8, 88.0, 84.3, 84.3, 83.4, 82.5, 78.9, 70.7]
bar_colors = [
    "#eaf1fe",  # blue-1
    "#eaf1fe",  # blue-1
    "#47484f",  # gray-5
    "#cedffe",  # blue-2
    "#47484f",  # gray-5
    "#5477c4",  # blue-4
    "#a3befa",  # blue-3
    "#47484f",  # gray-5
    "#47484f",  # gray-5
]

xy.bar_chart(
    *[
        xy.bar(
            x=[m], y=[s], color=c, width=0.85, corner_radius=8, stroke="#f0f1f2", stroke_width=1.5
        )
        for m, s, c in zip(models, scores, bar_colors, strict=True)
    ],
    *[
        xy.text(
            m,
            s,
            f"{s:.1f}%",
            dx=0,
            dy=-8,
            anchor="middle",
            color="#ffffff",
            style={"font_size": "15px", "font_weight": "500"},
        )
        for m, s in zip(models, scores, strict=True)
    ],
    # tick_label_anchor="end" pins each slanted name's trailing edge at its
    # tick (mpl ha='right'), so labels hang below the axis like the reference.
    xy.x_axis(label=None, tick_label_angle=-30, tick_label_anchor="end", style={"tick_length": 6}),
    xy.y_axis(
        label="Score",
        domain=(0, 100),
        style={"tick_length": 6},
        tick_values=[0, 25, 50, 75, 100],
        tick_labels=["0%", "25%", "50%", "75%", "100%"],
    ),
    xy.legend(show=False),
    # This is a dashboard summary, so keep its intended viewport fixed.
    # pan=False blocks drag-panning; zoom=False blocks wheel/box zoom and reset.
    xy.interaction_config(pan=False, zoom=False),
    # background= is the figure background (mpl figure.facecolor): it paints
    # the whole card — margins, title, tick labels — and shows through the
    # plot rect, so no separate plot_background is needed for a flat card.
    xy.theme(
        background="#000000",
        text_color="#ffffff",
        grid_color="transparent",
        axis_color="#f0f1f2",
    ),
    styles={
        "title": {
            "text_align": "left",
            "padding_left": "40px",
            "padding_top": "18px",
            "font_size": "22px",
            "font_weight": "700",
        },
        "tick_label": {"font_size": "13px"},
        "axis_title": {"font_size": "15px"},
    },
    # width="100%" fills the notebook output row - VS Code paints a white
    # backdrop behind widget outputs, so a fixed-width card would sit on white.
    title="TerminalBench 2.1",
    width="100%",
    height=680,
    padding=[90, 40, 160, 90],
)

## Reference recreation — ExploitBench (dark line-and-marker benchmark)

Five model series as **line + scatter pairs**: `xy.line` has no `marker=`, so the
dots are a separate `xy.scatter` per series; the scatter carries the `name`, so
legend entries get dot swatches and the unnamed lines stay out of the legend.
The dotted frontier rules are `xy.hline` with `style={"dash": ...}`. Rule labels
have no anchor control (they pin start-anchored at the plot's right edge and run
outward, clipping at the figure edge), so the frontier names are separate
end-anchored `xy.text` annotations pinned just inside the right edge, lifted above
each line with `dy`. The two single-model pins are `xy.marker`
(`symbol="diamond"` / `"square"`) with side text.

**Known gaps vs the reference (not worked around):** xy's legend `loc` only
anchors *inside* the plot rect — there is no outside-the-axes placement, so the
legend sits at `upper center` in the plot instead of above it; and there is no
image annotation, so the brand logo in the top-right corner is dropped.


In [ ]:
K = 1_000.0

series = [
    ("GPT-5.6 Sol", "#eaf1fe", [(30, 28.0), (62, 52.0), (95, 63.0), (105, 70.5), (118, 74.0)]),
    ("GPT-5.6 Terra", "#cedffe", [(32, 22.0), (105, 41.5), (135, 46.0), (162, 53.0)]),
    ("GPT-5.6 Luna", "#a3befa", [(35, 17.0), (80, 23.5), (108, 30.5)]),
    ("GPT-5.5", "#f6c3cb", [(55, 28.0), (88, 40.3), (98, 47.8), (120, 48.0)]),
    ("GPT-5.4", "#ef6fa8", [(85, 26.5), (125, 30.8), (135, 31.2), (160, 33.5), (200, 38.0)]),
]

marks = []
for name, color, pts in series:
    xs = [p[0] * K for p in pts]
    ys = [p[1] for p in pts]
    marks.append(xy.line(x=xs, y=ys, color=color, width=3))
    marks.append(xy.scatter(x=xs, y=ys, color=color, size=10, opacity=1.0, name=name))

# Rule labels have no anchor control, so the frontier names are end-anchored
# text annotations pinned just inside the plot's right edge, lifted clear of
# each dotted line with dy.
ref_label_style = {"font_size": "17px", "font_weight": "700"}
pin_style = {"vertical_align": "middle", "font_size": "17px", "font_weight": "700"}

chart = xy.scatter_chart(
    *marks,
    xy.hline(78.2, color="#f6e8cd", width=3, style={"dash": "2.5,9"}),
    xy.hline(40.3, color="#f2a07b", width=3, style={"dash": "2.5,9"}),
    xy.text(
        548 * K, 78.2, "Mythos 5", dy=-14, anchor="end", color="#f6e8cd", style=ref_label_style
    ),
    xy.text(
        548 * K, 40.3, "Opus 4.8", dy=-14, anchor="end", color="#f2a07b", style=ref_label_style
    ),
    xy.marker(
        310 * K,
        74.0,
        text="Mythos Preview",
        color="#f6e8cd",
        symbol="diamond",
        size=14,
        stroke_width=0,
        dx=18,
        dy=0,
        style=pin_style,
    ),
    xy.marker(
        225 * K,
        28.5,
        text="Opus 4.7",
        color="#ef8a51",
        symbol="square",
        size=13,
        stroke_width=0,
        dx=18,
        dy=0,
        style=pin_style,
    ),
    xy.x_axis(
        label="Output tokens",
        domain=(0, 560 * K),
        tick_values=[100 * K, 200 * K, 300 * K, 400 * K, 500 * K],
        tick_labels=["100K", "200K", "300K", "400K", "500K"],
        style={"tick_length": 6},
    ),
    xy.y_axis(
        label="Cap percent",
        domain=(12, 84),
        tick_values=[20, 40, 60, 80],
        tick_labels=["20%", "40%", "60%", "80%"],
        style={"tick_length": 6},
    ),
    # loc anchors inside the plot rect; the reference's above-the-axes legend
    # placement doesn't exist in xy yet (see the markdown cell above).
    xy.legend(loc="upper center", ncols=3),
    xy.theme(
        background="#000000",
        text_color="#ffffff",
        grid_color="transparent",
        axis_color="#f0f1f2",
    ),
    styles={
        "title": {
            "text_align": "left",
            "padding_left": "60px",
            "padding_top": "40px",
            "font_size": "24px",
            "font_weight": "700",
        },
        "tick_label": {"font_size": "16px", "font_family": "ui-monospace, monospace"},
        "axis_title": {"font_size": "17px"},
        "legend": {"font_size": "17px"},
    },
    title="ExploitBench",
    width=1160,
    height=1150,
    padding=[100, 60, 110, 110],
)
chart

In [ ]:
# PR check: a symlog axis keeps zero and negative values visible while
# compressing long positive and negative tails.
sample = np.arange(11)
signed_values = np.array(
    [-1_000_000, -100_000, -10_000, -1_000, -100, 0, 100, 1_000, 10_000, 100_000, 1_000_000]
)

symlog_chart = xy.scatter_chart(
    xy.line(x=sample, y=signed_values, color="#7dd3fc", width=2),
    xy.scatter(
        x=sample,
        y=signed_values,
        color="#f8fafc",
        size=10,
        opacity=1.0,
        name="Signed value",
    ),
    xy.hline(0, color="#fbbf24", width=2, style={"dash": "5,5"}),
    xy.x_axis(
        label="Sample",
        tick_values=sample.tolist(),
        tick_labels=[str(value) for value in signed_values],
        tick_label_angle=-35,
        tick_label_anchor="end",
    ),
    xy.y_axis(
        label="Signed value (symlog)",
        type_="symlog",
        constant=1_000,
        domain=(-1_000_000, 1_000_000),
        tick_values=[-1_000_000, -100_000, -10_000, -1_000, 0, 1_000, 10_000, 100_000, 1_000_000],
        tick_labels=["-1M", "-100K", "-10K", "-1K", "0", "1K", "10K", "100K", "1M"],
    ),
    xy.legend(show=False),
    xy.theme(
        background="#020617",
        text_color="#f8fafc",
        grid_color="#334155",
        axis_color="#94a3b8",
    ),
    title="Zero-inclusive long tails with a native symlog axis",
    width=1000,
    height=620,
    padding=[70, 50, 120, 100],
)
symlog_chart

In [ ]:
# Regression demo for #87: open Zoom controls, choose Box zoom, then
# box-zoom into the cloud three times and zoom back out. The point cloud
# should keep its two-dimensional spread instead of collapsing to a line.
precision_rng = np.random.default_rng(42)
precision_x = precision_rng.normal(0, 1, 20_000)
precision_y = 0.5 * precision_x + precision_rng.normal(0, 0.6, precision_x.size)
precision_color = precision_x**2 + precision_y**2
precision_size = np.abs(precision_rng.normal(size=precision_x.size))

xy.scatter_chart(
    xy.scatter(
        x=precision_x,
        y=precision_y,
        color=precision_color,
        size=precision_size,
        colormap="viridis",
        size_range=(3, 18),
        opacity=0.7,
    ),
    title="Repeated box zoom — axes must not overshoot home",
    width=900,
    height=500,
)

In [ ]:
# PR verification: the initial viewport is smaller than the hard navigation bounds.
# Drag and wheel-zoom aggressively; neither axis should move beyond the red rules.
viewport_x = np.linspace(0, 100, 501)
viewport_y = np.sin(viewport_x / 7) + 0.15 * np.cos(viewport_x / 2)

xy.line_chart(
    xy.line(viewport_x, viewport_y, name="signal", color="#5477c4", width=3),
    xy.vline(0, text="x min", color="#ef4444", width=2),
    xy.vline(100, text="x max", color="#ef4444", width=2),
    xy.hline(-1.25, text="y min", color="#ef4444", width=2),
    xy.hline(1.25, text="y max", color="#ef4444", width=2),
    xy.x_axis(label="x (hard bounds: 0 to 100)", domain=(30, 60), bounds=(0, 100)),
    xy.y_axis(
        label="signal (hard bounds: -1.25 to 1.25)", domain=(-0.6, 0.6), bounds=(-1.25, 1.25)
    ),
    xy.legend(show=False),
    title="Hard viewport bounds — pan and zoom cannot cross the red limits",
    width=1000,
    height=520,
)

## Regression check — linked facet interactions

This exercises the linked navigation and selection added by this branch. Drag or wheel-zoom inside one panel: every panel should follow on both axes. Shift-drag a box around points in one panel: the same data-space region should be selected in every panel. Double-click to reset the view and clear the linked selection.

In [ ]:
linked_x = np.linspace(0, 10, 50)
linked_facet_data = {
    "x": np.tile(linked_x, 3),
    "y": np.concatenate([np.sin(linked_x + phase) + 0.15 * phase for phase in (0.0, 0.6, 1.2)]),
    "group": np.repeat(["alpha", "beta", "gamma"], linked_x.size),
}

xy.facet_chart(
    xy.scatter(x="x", y="y", color="#5477c4", size=8),
    by="group",
    data=linked_facet_data,
    cols=3,
    link=True,
    link_select=True,
    title="Pan, zoom, and Shift-drag selection are linked",
    width=1080,
    height=320,
)